In [15]:
# filter_synthetic.py

import os
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import models, transforms


# --- CONFIG ---
SSL_CKPT = '/kaggle/input/datasets/ittisamurtunib/ssl-encoder-real-24000/ssl_encoder_real_24000.pth'  # Stage-1 checkpoint
REAL_MEL_CSV = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs/expA_train.csv'          # the split file with real MEL images
REAL_IMG_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'     # folder with real images
SYNTH_DIR = '/kaggle/input/datasets/ittisamurtunib/synthetic-mel-gan-dataset/outputs/synthetic_mel'       # folder with GAN images
SYNTH_MANIFEST = '/kaggle/input/datasets/ittisamurtunib/synthetic-mel-gan-dataset/outputs/synthetic_mel_manifest.csv' 

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- DEFINE ARCHITECTURES TO MATCH CHECKPOINT ---
class SimpleEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 3
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Pooling
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        h = self.features(x)
        z = self.fc(h)
        return z

class SimCLR(nn.Module):
    def __init__(self, encoder, projection_dim=64):
        super().__init__()
        self.encoder = encoder
        self.projector = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, projection_dim)
        )
    def forward(self, x):
        h = self.encoder(x)
        z = F.normalize(self.projector(h), dim=1)
        return h, z

# --- LOAD ENCODER ROBUSTLY ---
encoder = SimpleEncoder()
state = torch.load(SSL_CKPT, map_location=DEVICE)

# Clean up prefixes if saved from a wrapped model
new_state = {}
for k, v in state.items():
    k = k.replace('encoder.', '').replace('backbone.', '')
    new_state[k] = v

encoder.load_state_dict(new_state, strict=False)
encoder.to(DEVICE).eval()

# --- TRANSFORM (same as fine-tuning) ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def extract_features(img_paths):
    feats = []
    with torch.no_grad():
        for path in img_paths:
            img = Image.open(path).convert('RGB')
            tensor = transform(img).unsqueeze(0).to(DEVICE)
            feat = encoder(tensor).cpu()
            feats.append(F.normalize(feat, dim=1))
    return torch.cat(feats, dim=0)

# --- LOAD REAL MEL IMAGES ---
df_train = pd.read_csv(REAL_MEL_CSV)
real_mel_df = df_train[df_train['label'] == 'MEL']
real_paths = [os.path.join(REAL_IMG_DIR, f"{img}.jpg") for img in real_mel_df['image']]
real_feats = extract_features(real_paths)  # shape (35, 128)

# --- LOAD SYNTHETIC IMAGES ---
synth_df = pd.read_csv(SYNTH_MANIFEST)
synth_paths = [os.path.join(SYNTH_DIR, f"{row['image']}.jpg") for _, row in synth_df.iterrows()]
synth_feats = extract_features(synth_paths)  # shape (200, 128)

# --- COMPUTE NEAREST-NEIGHBOUR SIMILARITY ---
# cosine similarity: real_feats (35,128) x synth_feats.T (128,200)
sim_matrix = real_feats @ synth_feats.T  # (35, 200)
max_sim_per_synth, _ = sim_matrix.max(dim=0)  # best match for each synth

# --- FILTER ---
THRESHOLD = 0.90
keep_mask = (max_sim_per_synth < THRESHOLD).cpu().numpy()
kept_indices = synth_df.index[keep_mask].tolist()
rejected_indices = synth_df.index[~keep_mask].tolist()

print(f"Total synthetic: {len(synth_df)}")
print(f"Kept (diverse): {len(kept_indices)}")
print(f"Rejected (near-duplicates): {len(rejected_indices)}")

# Save the filtered list for the next training run
synth_df_kept = synth_df.iloc[kept_indices]
synth_df_kept.to_csv('outputs/synthetic_mel_filtered.csv', index=False)

Total synthetic: 200
Kept (diverse): 6
Rejected (near-duplicates): 194
